In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import openvino as ov
import librosa
import numpy as np
from transformers import AutoTokenizer
from IPython.display import Audio
import torch

/opt/conda/envs/sparktts/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openvino_export.bicodec import BiCodecTokenizer, BiCodecDetokenizer
from sparktts.models.bicodec import BiCodec
from openvino_export.wav2vec2 import Wav2Vec2Wrapper
from openvino_export.mel_spectrogram import MelSpectrogram
from optimum.intel.openvino import OVModelForCausalLM
from sparktts.utils.file import load_config

from openvino_tokenizers import convert_tokenizer

In [3]:
bicodec_config = load_config("./pretrained_models/Spark-TTS-0.5B/BiCodec/config.yaml")["audio_tokenizer"]
bicodec_config["mel_params"]

{'sample_rate': 16000, 'n_fft': 1024, 'win_length': 640, 'hop_length': 320, 'mel_fmin': 10, 'mel_fmax': None, 'num_mels': 128}

In [4]:
llm = OVModelForCausalLM.from_pretrained("./pretrained_models/Spark-TTS-0.5B/LLM")

No OpenVINO files were found for ./pretrained_models/Spark-TTS-0.5B/LLM, setting `export=True` to convert the model to the OpenVINO IR. Don't forget to save the resulting model with `.save_pretrained()`
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)
/opt/conda/envs/sparktts/lib/python3.12/site-packages/optimum/exporters/openvino/model_patcher.py:552: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if sequence_length != 1:


In [8]:
hf_llm_tokenizer = AutoTokenizer.from_pretrained("./pretrained_models/Spark-TTS-0.5B/LLM")
ov_tokenizer, ov_detokenizer = convert_tokenizer(hf_llm_tokenizer, with_detokenizer=True)

In [9]:
llm.save_pretrained("./openvino_models/Spark-TTS-0.5B/LLM")
ov.save_model(ov_tokenizer, "./openvino_models/Spark-TTS-0.5B/LLM/openvino_tokenizer.xml")
ov.save_model(ov_detokenizer, "./openvino_models/Spark-TTS-0.5B/LLM/openvino_detokenizer.xml")

In [ ]:
wav2vec = Wav2Vec2Wrapper("./pretrained_models/Spark-TTS-0.5B/wav2vec2-large-xlsr-53")
bicodec = BiCodec.load_from_checkpoint("./pretrained_models/Spark-TTS-0.5B/BiCodec")
mel_spectrogram = MelSpectrogram(bicodec_config["mel_params"])

In [ ]:
tokenizer = BiCodecTokenizer(bicodec)
detokenizer = BiCodecDetokenizer(bicodec)

tokenizer.eval()
detokenizer.eval()

In [ ]:
audio, sr = librosa.load("./example/prompt_audio.wav", sr=16000)
audio = librosa.util.normalize(audio)
# truncate/pad to 6s (16kHz) audio
audio = np.pad(audio, (0, max(0, 96000 - len(audio))), mode='constant')[:96000]
audio.shape, sr, audio.max(), audio.min(), audio.mean(), audio.std()

In [ ]:
feat_input = torch.tensor(audio).unsqueeze(0)
feat = wav2vec(feat_input)
feat.shape, feat.max(), feat.min(), feat_input.shape

In [ ]:
mel_input = torch.tensor(audio).unsqueeze(0).unsqueeze(0)
mel = mel_spectrogram(mel_input)
mel.shape, mel.max(), mel.min(), mel_input.shape

In [ ]:
semantic_tokens, global_tokens = tokenizer(feat, mel)
semantic_tokens.shape, global_tokens.shape, semantic_tokens.dtype, global_tokens.dtype

In [ ]:
# # truncate semantic tokens to 50 tokens(1s)
# semantic_tokens = semantic_tokens[:, 50:100]
# semantic_tokens.shape, semantic_tokens.max(), semantic_tokens.min()

In [ ]:
wav = detokenizer(semantic_tokens, global_tokens)
wav.shape, wav.max(), wav.min()

In [ ]:
Audio(audio, rate=sr)  # Play the audio to verify it loaded correctly

In [ ]:
cpu_wav = wav.detach().cpu().numpy().squeeze().squeeze()
Audio(cpu_wav, rate=sr)  # Play the detokenized audio to

In [ ]:
ov_mel_spectrogram = ov.convert_model(mel_spectrogram, example_input=mel_input)

In [ ]:
ov_wav2vec = ov.convert_model(wav2vec, example_input=feat_input)

In [ ]:
ov_tokenizer = ov.convert_model(tokenizer, example_input=(feat, mel))

In [ ]:
ov_detokenizer = ov.convert_model(detokenizer, example_input=(semantic_tokens, global_tokens))

In [ ]:
# Save the OpenVINO models
ov.save_model(ov_mel_spectrogram, "./openvino_models/Spark-TTS-0.5B/mel_spectrogram.xml")
ov.save_model(ov_wav2vec, "./openvino_models/Spark-TTS-0.5B/wav2vec.xml")
ov.save_model(ov_tokenizer, "./openvino_models/Spark-TTS-0.5B/tokenizer.xml")
ov.save_model(ov_detokenizer, "./openvino_models/Spark-TTS-0.5B/detokenizer.xml")

In [ ]:
# Compile the OpenVINO model
device_name = "CPU"
core = ov.Core()
ovc_mel_spectrogram = core.compile_model(ov_mel_spectrogram, device_name)
ovc_wav2vec = core.compile_model(ov_wav2vec, device_name)
ovc_tokenizer = core.compile_model(ov_tokenizer, device_name)
ovc_detokenizer = core.compile_model(ov_detokenizer, device_name)

In [ ]:
# inference with OpenVINO
ov_audio = librosa.load("./ref.wav", sr=16000)[0]
# make ov_audio is the same shape as audio
# pad or trim
if len(ov_audio) < len(audio):
    ov_audio = np.pad(ov_audio, (0, len(audio) - len(ov_audio)), mode='constant')
elif len(ov_audio) > len(audio):
    ov_audio = ov_audio[:len(audio)]
ov_audio = librosa.util.normalize(ov_audio)
ov_audio.shape, ov_audio.max(), ov_audio.min(), ov_audio.mean(), ov_audio.std()

In [ ]:
ov_mel_input = torch.tensor(ov_audio).unsqueeze(0).unsqueeze(0)
ov_mel = ovc_mel_spectrogram(ov_mel_input) 
ov_mel[0].shape, ov_mel[0].max(), ov_mel[0].min()


In [ ]:
ov_feat_input = torch.tensor(ov_audio).unsqueeze(0)
ov_feat = ovc_wav2vec(ov_feat_input)
ov_feat[0].shape, ov_feat[0].max(), ov_feat[0].min()

In [ ]:
ov_tokens = ovc_tokenizer((ov_feat[0], ov_mel[0]))

In [ ]:
ov_semantic_tokens = ov_tokens[0]
ov_global_tokens = ov_tokens[1]
ov_semantic_tokens.shape, ov_global_tokens.shape, ov_semantic_tokens.dtype, ov_global_tokens.dtype

In [ ]:
ov_wav = ovc_detokenizer((ov_semantic_tokens, ov_global_tokens))
ov_wav[0].shape, ov_wav[0].max(), ov_wav[0].min()

In [ ]:
Audio(ov_audio, rate=sr)  # Play the audio to verify it loaded correctly

In [ ]:
ov_cpu_wav = ov_wav[0].squeeze().squeeze()
Audio(ov_cpu_wav, rate=sr)  # Play the detokenized